# 04 — Prediction layer: next-month direction, honestly evaluated

**Phase 3.** The payoff of the locked explore/holdout split. Two targets:

1. **Primary:** next-month return *direction* (up/down) for each of the 8 core
   commodities. Not the return itself — monthly returns are noise-dominated.
2. **Secondary:** an elevated-volatility flag (volatility clusters, so this has
   real predictive content even where price direction doesn't).

**The honest bar:** a gradient-boosting model must beat a naive baseline
(persistence + majority class) *on the untouched holdout*. "Barely beats
baseline" is a valid, reportable result. The holdout (2022→present) is touched
exactly once, at the end.

In [ ]:
import sys
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
sys.path.insert(0, str(Path.cwd().parent))
import config
from src import predict
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, accuracy_score, brier_score_loss
from sklearn.calibration import calibration_curve
plt.rcParams["figure.figsize"] = (11,5)

monthly = pd.read_parquet(config.DATA / "commodities_monthly.parquet")
X = predict.build_features(monthly)
Y = predict.build_targets(monthly)
FEAT = list(X.columns)
print("features:", len(FEAT), "| targets:", list(Y.columns))

## Walk-forward on EXPLORE only (model selection)

Expanding-window walk-forward inside the explore period. This is how we choose
whether the GBM is worth anything, WITHOUT touching the holdout. Each step trains
on all months up to t, predicts t+1.

In [ ]:
def walk_forward(target, model_fn, min_train=120):
    d = X.join(Y[target]).loc[:config.EXPLORE_END].dropna(subset=FEAT+[target])
    d[target] = d[target].astype(int)
    preds, actuals = [], []
    for i in range(min_train, len(d)):
        tr = d.iloc[:i]; te = d.iloc[i:i+1]
        if tr[target].nunique() < 2:  # need both classes
            continue
        m = model_fn().fit(tr[FEAT], tr[target])
        preds.append(m.predict_proba(te[FEAT])[0,1])
        actuals.append(int(te[target].iloc[0]))
    return np.array(actuals), np.array(preds)

def gbm(): return HistGradientBoostingClassifier(max_iter=200, max_depth=3,
                 learning_rate=0.05, l2_regularization=1.0, random_state=0)
def logit(): return make_pipeline(StandardScaler(),
                 LogisticRegression(max_iter=1000, C=0.5))

# evaluate both model classes on gold, in-explore walk-forward
for name, fn in [("logistic", logit), ("gbm", gbm)]:
    a, p = walk_forward("gold_up", fn)
    print(f"gold  {name:9s} walk-forward AUC={roc_auc_score(a,p):.3f}  acc={accuracy_score(a,(p>0.5)):.3f}  n={len(a)}")

**Read:** if neither model clears ~0.53-0.55 AUC in walk-forward, there's
little directional signal and the holdout will confirm it. Pick the model class
that does better in-explore (likely GBM, but logistic is the honest floor).

## Explore walk-forward across all 8 — where is there any signal?

In [ ]:
rows=[]
for c in predict.CORE:
    a,p = walk_forward(f"{c}_up", gbm)
    base = a.mean(); base = max(base, 1-base)  # majority-class accuracy
    rows.append({"asset":c, "auc":roc_auc_score(a,p) if len(set(a))>1 else np.nan,
                 "acc":accuracy_score(a,(p>0.5)), "majority_base":base, "n":len(a)})
explore_scores = pd.DataFrame(rows).set_index("asset")
explore_scores.round(3)

## THE HOLDOUT — touched once

Train on ALL explore data, predict every month of 2022→present. Compare GBM vs
both baselines. This is the real result; nothing gets tuned after seeing it.

In [ ]:
def holdout_eval(target):
    tr = X.join(Y[target]).loc[:config.EXPLORE_END].dropna(subset=FEAT+[target])
    ho = X.join(Y[target]).loc[config.HOLDOUT_START:].dropna(subset=FEAT+[target])
    tr[target]=tr[target].astype(int); ho[target]=ho[target].astype(int)
    m = gbm().fit(tr[FEAT], tr[target])
    p = m.predict_proba(ho[FEAT])[:,1]
    y = ho[target].values
    auc = roc_auc_score(y,p) if len(set(y))>1 else np.nan
    acc = accuracy_score(y, (p>0.5))
    # baselines
    feat_ret = ho[f"{target.replace('_up','')}_ret"] if f"{target.replace('_up','')}_ret" in ho else None
    persist = predict.baseline_persistence(feat_ret, pd.Series(y, index=ho.index)) if feat_ret is not None else np.nan
    maj = predict.baseline_majority(tr[target], pd.Series(y))
    return {"auc":auc,"gbm_acc":acc,"persist_acc":persist,"majority_acc":maj,
            "brier":brier_score_loss(y,p) if len(set(y))>1 else np.nan,"n":len(y),"probs":p,"y":y}

hrows={}
for c in predict.CORE:
    r=holdout_eval(f"{c}_up"); hrows[c]=r
holdout = pd.DataFrame({c:{k:v for k,v in r.items() if k not in ("probs","y")}
                        for c,r in hrows.items()}).T
holdout[["auc","gbm_acc","persist_acc","majority_acc","brier","n"]].round(3)

**The verdict table.** For each asset compare `gbm_acc` against
`persist_acc` and `majority_acc`. GBM earns its place only where it beats BOTH
baselines and AUC > 0.5. Expect this to be unimpressive for most assets — that
IS the finding: monthly commodity direction is largely unpredictable from macro,
consistent with efficient-ish commodity markets.

In [ ]:
# aggregate honesty check: how often does GBM beat both baselines?
wins = ((holdout["gbm_acc"]>holdout["persist_acc"]) &
        (holdout["gbm_acc"]>holdout["majority_acc"]) &
        (holdout["auc"]>0.5)).sum()
print(f"GBM beats both baselines AND auc>0.5 in {wins} of 8 assets")
print(f"mean holdout AUC across assets: {holdout['auc'].mean():.3f}")
print(f"mean GBM acc: {holdout['gbm_acc'].mean():.3f}  vs majority: {holdout['majority_acc'].mean():.3f}")

## Calibration — are the probabilities honest?

In [ ]:
# pool all assets' holdout predictions for a calibration curve
allp = np.concatenate([hrows[c]["probs"] for c in predict.CORE])
ally = np.concatenate([hrows[c]["y"] for c in predict.CORE])
frac, mean_pred = calibration_curve(ally, allp, n_bins=8, strategy="quantile")
fig,ax=plt.subplots(figsize=(6,6))
ax.plot([0,1],[0,1],"k--",lw=1,label="perfect")
ax.plot(mean_pred,frac,"o-",color="steelblue",label="GBM (pooled)")
ax.set_xlabel("predicted P(up)"); ax.set_ylabel("observed freq up")
ax.set_title("Holdout calibration (all assets pooled)"); ax.legend(); plt.tight_layout(); plt.show()

## Secondary target — the volatility regime flag

In [ ]:
r = holdout_eval("vol_elevated")
print(f"VOL-REGIME holdout:  AUC={r['auc']:.3f}  acc={r['gbm_acc']:.3f}  "
      f"majority={r['majority_acc']:.3f}  n={r['n']}")
print("\nIf this AUC is clearly >0.5 and beats majority, the vol flag has real")
print("predictive content -- the dashboard's most honest predictive feature,")
print("even where price direction does not.")

## Verdict (fill after running)

- **Direction:** in how many of 8 assets does GBM beat both baselines? Mean AUC?
- **Calibration:** does the curve track the diagonal, or is the model over/under-confident?
- **Vol regime:** does the secondary target carry real signal?

**Likely honest conclusion:** monthly price direction is near-unpredictable
(GBM ~ baselines, AUC ~0.5); the vol-regime flag is the one genuinely useful
predictive output. That's a defensible, dashboard-shaping result — the trader
gets a calibrated volatility-context signal, not false-precision price calls.